# Data Challenge: Cargo Volume
- We are tasked to predict cargo colume through San Francisco International Airport

In [ ]:
import pandas as pd

train_df = pd.read_csv('Cargo Volume - Train.csv')
test_df = pd.read_csv('Cargo Volume - Test.csv')
train_df.head()

In [ ]:
train_df.info()

In [ ]:
test_df.info()

# Data Analysis
- We can see the majority of our dataset consists of Categorical data thus some type of encoding must take place.
- It should also be noted that the column 'Activity Period' is in a format that will eventually involve some Feature Engineering.
- From the project spec we expected the target column 'Cargo Metric TONS' to be missing from our test data set.

In [ ]:
train_df.isna().sum()

In [ ]:
train_df = train_df.dropna()

In [ ]:
test_df.isna().sum()

In [ ]:
test_df = test_df.dropna()

In [ ]:
train_df.describe(include='object')

In [ ]:
train_df.nunique()

# Feature Engineering: Activity Period
- There are monthly rhythms for all air travel, thus it would benefit us to utilize this data.
- Sliceing the yyyymm formate into separate columns should increase model accuracy.
- We need to remember to implement these transformations on our test dataframe.

In [ ]:
train_df['Year'] = train_df['Activity Period'].astype(str).str.slice(0, 4).astype(int)
train_df['Month'] = train_df['Activity Period'].astype(str).str.slice(4, 6).astype(int)

train_df.columns

In [ ]:
test_df['Year'] = test_df['Activity Period'].astype(str).str.slice(0, 4).astype(int)
test_df['Month'] = test_df['Activity Period'].astype(str).str.slice(4, 6).astype(int)

test_df.columns

# Feature Selection
- Since our values are nominal, label encoding would not be an ideal way to encode our data.
- The usual way to encode nominal variables is with a one-hot-encoding.
- However, the columns with large amounts of unique values could introduce the curse of dimensionality.
- Because of this I'd like to eliminate redundacy

In [ ]:
print(train_df['Operating Airline'].unique())
print()
print(train_df['Published Airline'].unique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encoding so that we can check the correlation
le_operating = LabelEncoder()
le_published = LabelEncoder()

train_df['Operating Airline Encoded'] = le_operating.fit_transform(train_df['Operating Airline'])
train_df['Published Airline Encoded'] = le_published.fit_transform(train_df['Published Airline'])

# Apply Spearman's correlation method
operating_corr = train_df['Operating Airline Encoded'].corr(train_df['Cargo Metric TONS'], method='spearman')
published_corr = train_df['Published Airline Encoded'].corr(train_df['Cargo Metric TONS'], method='spearman')

print(f"Correlation with Cargo Metric TONS : Operating Airline: {operating_corr}")
print(f"Correlation with Cargo Metric TONS : Published Airline: {published_corr}")

train_df.drop(['Operating Airline Encoded', 'Published Airline Encoded'], axis=1, inplace=True)

In [ ]:
print(train_df['Operating Airline IATA Code'].unique())
print()
print(train_df['Published Airline IATA Code'].unique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encoding so that I can check the correlation
le_operating = LabelEncoder()
le_published = LabelEncoder()

train_df['Operating Airline IATA Code Encoded'] = le_operating.fit_transform(train_df['Operating Airline IATA Code'])
train_df['Published Airline IATA Code Encoded'] = le_published.fit_transform(train_df['Published Airline IATA Code'])

# Apply Spearman's correlation method 
operating_corr = train_df['Operating Airline IATA Code Encoded'].corr(train_df['Cargo Metric TONS'], method='spearman')
published_corr = train_df['Published Airline IATA Code Encoded'].corr(train_df['Cargo Metric TONS'], method='spearman')

print(f"Correlation with Cargo Metric TONS : Operating Airline IATA Code: {operating_corr}")
print(f"Correlation with Cargo Metric TONS : Published Airline IATA Code: {published_corr}")

train_df.drop(['Operating Airline IATA Code Encoded', 'Published Airline IATA Code Encoded'], axis=1, inplace=True)

# Findings
- All four columns from the dataset show very weak correlations to the target.
- With this information I'll remove the weakest correlations in an effort to reduce dimensonality.

In [ ]:
train_df.drop(['Published Airline', 'Published Airline IATA Code', 'Activity Period'], axis=1, inplace=True)
test_df.drop(['Published Airline', 'Published Airline IATA Code', 'Activity Period'], axis=1, inplace=True)

In [ ]:
train_df.columns

In [ ]:
test_df.columns

# Feature Engineering: Month
- To improve the modeling process we'll augment the Month column into sine and cosine.
- This will highlight any cyclical patterns in the dataset

In [ ]:
import numpy as np

train_df['Month_sin'] = np.sin(train_df.Month * 2 * np.pi / len(train_df.Month.unique()))
train_df['Month_cos'] = np.cos(train_df.Month * 2 * np.pi / len(train_df.Month.unique()))

cols = train_df.columns.tolist()
cols.remove('Cargo Metric TONS')
cols.append('Cargo Metric TONS')

train_df = train_df[cols]

train_df.drop(['Month'], axis=1, inplace=True)
train_df.head()

In [ ]:
test_df['Month_sin'] = np.sin(test_df.Month * 2 * np.pi / len(test_df.Month.unique()))
test_df['Month_cos'] = np.cos(test_df.Month * 2 * np.pi / len(test_df.Month.unique()))

test_df.drop(['Month'], axis=1, inplace=True)
test_df.head()

# Ecoding
- As previously mentioned we are working with categorical values with no intrinsic ordering.
- Thus we will one-hot-encode the strings in our dataset.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = {}
cols_to_enc = ['Operating Airline', 'Operating Airline IATA Code', 'GEO Summary', 
               'GEO Region', 'Activity Type Code', 'Cargo Type Code', 'Cargo Aircraft Type']

label_train_df = train_df.copy()
label_test_df = test_df.copy()

for col in cols_to_enc:
    label_enc[col] = LabelEncoder()
    
    combined_data = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    label_enc[col].fit(combined_data)
    label_train_df[col] = label_enc[col].transform(train_df[col].astype(str))
    
    label_test_df[col] = label_enc[col].transform(test_df[col].astype(str))

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder(sparse_output=False)

combined_data = pd.concat([label_train_df[cols_to_enc], label_test_df[cols_to_enc]], axis=0)
one_hot_encoder.fit(combined_data)

one_hot_encoded_train = one_hot_encoder.transform(label_train_df[cols_to_enc])
encoded_train_df = pd.DataFrame(one_hot_encoded_train,
                                columns=one_hot_encoder.get_feature_names_out(cols_to_enc),
                                index=label_train_df.index)

one_hot_encoded_test = one_hot_encoder.transform(label_test_df[cols_to_enc])
encoded_test_df = pd.DataFrame(one_hot_encoded_test,
                               columns=one_hot_encoder.get_feature_names_out(cols_to_enc),
                               index=label_test_df.index)

encoded_train_df = pd.concat([label_train_df.drop(cols_to_enc, axis=1), encoded_train_df], axis=1)
encoded_test_df = pd.concat([label_test_df.drop(cols_to_enc, axis=1), encoded_test_df], axis=1)

# LinearRegression
- Linear Regression is dependent on scale.
- This means our one-hot-encoded values could cause issues.
- I'll use the label-encoded DataFrame to train.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

lin_reg = LinearRegression()

X = label_train_df.drop('Cargo Metric TONS', axis=1)
y = label_train_df['Cargo Metric TONS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

lin_reg.fit(X_train, y_train)
lin_mse = mean_squared_error(y_test, lin_reg.predict(X_test))
print(f'LinearRegression RMSE: {np.sqrt(lin_mse)}')

# Lasso
- Lasso uses the alpha term to prevent overfitting.
- Thus using the one-hot-encoded values should cause adverse effects.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV

lasso = Lasso(alpha=1)

X = encoded_train_df.drop('Cargo Metric TONS', axis=1)
y = encoded_train_df['Cargo Metric TONS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

lasso.fit(X_train, y_train)
lasso_mse = mean_squared_error(y_test, lasso.predict(X_test))
print(f'Lasso RMSE: {np.sqrt(lasso_mse)}')

# Ridge
- Ridge is another regression model that uses alpha to prevent overfitting.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1)

X = encoded_train_df.drop('Cargo Metric TONS', axis=1)
y = encoded_train_df['Cargo Metric TONS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

ridge.fit(X_train, y_train)
ridge_mse = mean_squared_error(y_test, ridge.predict(X_test))
print(f'Ridge RMSE: {np.sqrt(ridge_mse)}')

# CART
- CART decision trees handle outlines well.
- This has been one of my most consistent models.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt_reg = DecisionTreeRegressor(criterion='friedman_mse',
                               max_depth=10,
                               min_samples_leaf=2,
                               min_samples_split=15)

X = encoded_train_df.drop('Cargo Metric TONS', axis=1)
y = encoded_train_df['Cargo Metric TONS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

dt_reg.fit(X_train, y_train)

dt_mse = mean_squared_error(y_test, dt_reg.predict(X_test))
print(f'CART RMSE: {np.sqrt(dt_mse)}')

# XGBoost
- Because we one-hot-encoded two columns xgb is an ideal model beacuse of how well it works on large ds.
- Since this Data Challenge is one envolving a regression, xgb's gradient boosting framework is ideal.

In [ ]:
import xgboost as xgb

xg_reg = xgb.XGBRegressor(colsample_bytree = 0.814, 
                          learning_rate = 0.049,
                          max_depth = 9,  
                          n_estimators = 293,
                          n_jobs=-1,
                          reg_lambda=3,
                          reg_alpha=1)

X = encoded_train_df.drop('Cargo Metric TONS', axis=1)
y = encoded_train_df['Cargo Metric TONS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

xg_reg.fit(X_train, y_train)
xg_mse = mean_squared_error(y_test, xg_reg.predict(X_test))
print(f'XGBoost RMSE: {np.sqrt(xg_mse)}')

# RandomForest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(max_depth=30,
                               n_estimators=50,
                               n_jobs=-1,
                               min_samples_leaf=4)

X = encoded_train_df.drop('Cargo Metric TONS', axis=1)
y = encoded_train_df['Cargo Metric TONS']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

rf_reg.fit(X_train, y_train)
rf_mse = mean_squared_error(y_test, rf_reg.predict(X_test))
print(f'Random Forest RMSE: {np.sqrt(rf_mse)}')

# Choosing a model : RandomForest
- Since hyperparameter tuning RandomForest and XGBoost they have produced very similar RMSE scores.
- I was going to use k-fold to compare the performace of the two one last time but have decided against it.
- LinearRegression, Lasso, and Ridge are all linear models.
- Because all three of those models preformed the worst it suggest that either non-linearity or high dimensionality came into play.
- As for why RandomForest bested XGBoost I'm not sure.
- RandomForest's may have pickup up on the seasonal features of the engineered Month columns better than XGBoost.

In [ ]:
X = encoded_test_df
y_target_col = rf_reg.predict(X)
y_target_col_df = pd.DataFrame({'Cargo Metric TONS': y_target_col})
y_target_col_df.to_csv('dc1.csv', index=False)